<a href="https://colab.research.google.com/github/jkj05/BIS5/blob/main/GWO_%26_PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:

import random
import math

WOLVES = 30
ITERATIONS = 100
STATIONS = 4
LOW, HIGH = 0, 1000

targets = [(x, y) for x in range(200, 801, 150)
           for y in range(200, 801, 150)]

def fitness(wolf):
    stations = [(wolf[i], wolf[i + 1])
                for i in range(0, len(wolf), 2)]

    error = 0

    for x, y in targets:
        distances = [
            math.hypot(x - sx, y - sy)
            for sx, sy in stations
        ]

        gradients = []

        for i in range(1, STATIONS):
            gradients.append((
                (x - stations[i][0]) / max(distances[i], 1e-9)
                - (x - stations[0][0]) / max(distances[0], 1e-9),
                (y - stations[i][1]) / max(distances[i], 1e-9)
                - (y - stations[0][1]) / max(distances[0], 1e-9)
            ))

        sx = sum(g[0] for g in gradients)
        sy = sum(g[1] for g in gradients)

        fxx = (sum(g[0] ** 2 for g in gradients) - sx ** 2 / STATIONS)
        fyy = (sum(g[1] ** 2 for g in gradients) - sy ** 2 / STATIONS)
        fxy = (sum(g[0] * g[1] for g in gradients) - sx * sy / STATIONS)

        determinant = fxx * fyy - fxy ** 2

        if determinant <= 1e-12:
            return float('inf')

        error += (fxx + fyy) / determinant

    return error / len(targets)


wolves = [
    [random.uniform(LOW, HIGH) for _ in range(STATIONS * 2)]
    for _ in range(WOLVES)
]

best_solution = None
best_error = float('inf')

for iteration in range(ITERATIONS):
    wolves.sort(key=fitness)

    alpha = wolves[0][:]
    beta = wolves[1][:]
    delta = wolves[2][:]

    if fitness(alpha) < best_error:
        best_solution = alpha[:]
        best_error = fitness(alpha)

    a = 2 - 2 * iteration / ITERATIONS

    updated_wolves = []

    for wolf in wolves:
        new_wolf = []

        for j in range(STATIONS * 2):
            positions = []

            for leader in (alpha, beta, delta):
                r1, r2 = random.random(), random.random()
                A = 2 * a * r1 - a
                C = 2 * r2

                D = abs(C * leader[j] - wolf[j])
                positions.append(leader[j] - A * D)

            new_position = sum(positions) / 3
            new_wolf.append(max(LOW, min(HIGH, new_position)))

        updated_wolves.append(new_wolf)

    wolves = updated_wolves

print("\nOptimal Receiving Station Positions:")

for i in range(STATIONS):
    x, y = best_solution[2 * i:2 * i + 2]
    print(f"Station {i + 1}: ({x:.2f}, {y:.2f})")

print(f"\nMinimum Regional Localization Error: {best_error:.6f}")



Optimal Receiving Station Positions:
Station 1: (1000.00, 3.72)
Station 2: (10.11, 1000.00)
Station 3: (9.28, 4.87)
Station 4: (1000.00, 1000.00)

Minimum Regional Localization Error: 1.078378


In [13]:

import random

MONTHS = 12
CELLS = 25
ITERATIONS = 200

CAPACITY = 500
MIN_STORAGE = 50
INITIAL_STORAGE = 300
INSTALLED_POWER = 250

inflow = [80, 70, 60, 40, 30, 20, 25, 35, 50, 65, 85, 90]
demand = [55, 55, 60, 65, 70, 75, 80, 80, 75, 65, 60, 55]


def evaluate(schedule):
    storage = INITIAL_STORAGE
    supply_error = 0
    energy_error = 0
    releases = []
    power_output = []

    for month in range(MONTHS):
        available = max(0, storage + inflow[month] - MIN_STORAGE)
        release = min(max(0, schedule[month]), available)

        storage = min(CAPACITY, storage + inflow[month] - release)

        head = 30 + 20 * storage / CAPACITY
        power = 0.08 * release * head

        supply_error += (release - demand[month]) ** 2
        energy_error += max(0, INSTALLED_POWER - power) ** 2

        releases.append(release)
        power_output.append(power)

    return supply_error, energy_error, releases, power_output


def initialize():
    return [
        [random.uniform(10, 100) for _ in range(MONTHS)]
        for _ in range(CELLS)
    ]


def update_ca(population, objective):
    updated = []

    for i in range(CELLS):
        neighbors = [
            population[(i - 1) % CELLS],
            population[i],
            population[(i + 1) % CELLS]
        ]

        best_neighbor = min(
            neighbors,
            key=lambda s: evaluate(s)[objective]
        )

        new_schedule = [
            max(0, min(120,
                0.7 * population[i][m]
                + 0.3 * best_neighbor[m]
                + random.uniform(-8, 8)
            ))
            for m in range(MONTHS)
        ]

        updated.append(new_schedule)

    return updated


water_ca = initialize()
energy_ca = initialize()

best_water = None
best_energy = None

for iteration in range(ITERATIONS):
    water_best = min(water_ca, key=lambda s: evaluate(s)[0])
    energy_best = min(energy_ca, key=lambda s: evaluate(s)[1])

    if best_water is None or evaluate(water_best)[0] < evaluate(best_water)[0]:
        best_water = water_best[:]

    if best_energy is None or evaluate(energy_best)[1] < evaluate(best_energy)[1]:
        best_energy = energy_best[:]

    water_ca[water_ca.index(max(water_ca, key=lambda s: evaluate(s)[0]))] = energy_best[:]
    energy_ca[energy_ca.index(max(energy_ca, key=lambda s: evaluate(s)[1]))] = water_best[:]

    water_ca = update_ca(water_ca, 0)
    energy_ca = update_ca(energy_ca, 1)


print("\nBest Water Supply Schedule:")
supply, energy, releases, power = evaluate(best_water)

print("Monthly Releases:", [round(x, 2) for x in releases])
print(f"Water Supply Deviation: {supply:.2f}")
print(f"Hydropower Deficit: {energy:.2f}")

print("\nBest Hydropower Schedule:")
supply, energy, releases, power = evaluate(best_energy)

print("Monthly Releases:", [round(x, 2) for x in releases])
print(f"Water Supply Deviation: {supply:.2f}")
print(f"Hydropower Deficit: {energy:.2f}")

print("\nMonthly Hydropower Production:")
print([round(x, 2) for x in power])



Best Water Supply Schedule:
Monthly Releases: [56.93, 53.91, 68.41, 61.46, 73.64, 72.55, 82.96, 83.57, 75.36, 67.76, 54.97, 55.51]
Water Supply Deviation: 162.20
Hydropower Deficit: 35646.29

Best Hydropower Schedule:
Monthly Releases: [64.93, 65.13, 59.6, 63.99, 73.83, 67.03, 79.05, 77.7, 89.07, 84.67, 85.0, 90.0]
Water Supply Deviation: 2721.66
Hydropower Deficit: 11757.70

Monthly Hydropower Production:
[221.29, 222.98, 204.12, 214.27, 236.85, 204.96, 228.03, 213.53, 233.64, 216.76, 217.6, 230.4]
